In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/MINI_PROJECT/DAiSEE/project")
print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/.shortcut-targets-by-id/1juEhlfoTVwLCaEKw1JRlOUwEKEUhl0Or/MINI_PROJECT/DAiSEE/project


In [ ]:
!cp -r /content/drive/MyDrive/MINI_PROJECT/DAiSEE/project/data /content/

In [3]:
import pandas as pd

train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")

print(len(train_df), len(val_df), len(test_df))

os.makedirs("src", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)


4174 895 895


In [4]:
%%writefile src/dataset.py
import os
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image
import torch

class EngagementDataset(Dataset):

    def __init__(self, csv_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform
        self.root = "/content"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]["image_path"]
        label = self.data.iloc[idx]["label"] - 1
        img_path = os.path.join(self.root, img_path)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label)


Overwriting src/dataset.py


In [5]:
%%writefile src/model_v2.py
import torch.nn as nn
import torchvision.models as models

class EfficientNetB2(nn.Module):

    def __init__(self):
        super().__init__()
        model = models.efficientnet_b2(weights="IMAGENET1K_V1")
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(1408, 3)
        )
        self.model = model

    def forward(self, x):
        return self.model(x)


Overwriting src/model_v2.py


In [6]:
%%writefile src/train_v2.py
import torch
import pandas as pd
import numpy as np

from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.utils.class_weight import compute_class_weight

from src.dataset import EngagementDataset
from src.model_v2 import EfficientNetB2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = EngagementDataset("/content/data/train.csv", train_transform)
val_dataset   = EngagementDataset("/content/data/val.csv",   val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

train_df = pd.read_csv("data/train.csv")
labels = train_df["label"].values - 1

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:", class_weights)

def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, pred = torch.max(outputs, 1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return correct / total

model     = EfficientNetB2().to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=4
)

best = 0
EPOCHS = 50

print(f"\nStarting training for {EPOCHS} epochs...\n")

for epoch in range(EPOCHS):
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    acc = evaluate(model)
    scheduler.step(acc)

    print(f"Epoch: {epoch+1:2d} | Val Accuracy: {acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if acc > best:
        best = acc
        # saves to a NEW file, old final_model.pth is untouched
        torch.save(model.state_dict(), "models/final_model_v2.pth")
        print(f"  --> Best model saved (val acc: {best:.4f})")

print(f"\nTraining Completed | Best Validation Accuracy: {best:.4f}")


Overwriting src/train_v2.py


In [7]:
%%writefile src/evaluate_v2.py
import os
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report

from src.dataset import EngagementDataset
from src.model_v2 import EfficientNetB2

os.makedirs("results", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_dataset = EngagementDataset("/content/data/test.csv", transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    num_workers=2,
    pin_memory=True
)

model = EfficientNetB2()
model.load_state_dict(torch.load("models/final_model_v2.pth", map_location=device))
model = model.to(device)
model.eval()

pred = []
true = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, p = torch.max(outputs, 1)
        pred.extend(p.cpu().numpy())
        true.extend(labels.numpy())

accuracy = np.mean(np.array(true) == np.array(pred))
print("Test Accuracy:", accuracy)

# saves to new file, old results untouched
with open("results/test_accuracy_v2.txt", "w") as f:
    f.write(str(accuracy))

cm = confusion_matrix(true, pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low","Medium","High"],
            yticklabels=["Low","Medium","High"])
plt.title("Confusion Matrix - V2")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.savefig("results/confusion_matrix_v2.png")
plt.close()

report = classification_report(true, pred,
                                target_names=["Low","Medium","High"])
print(report)

with open("results/classification_report_v2.txt", "w") as f:
    f.write(report)


Overwriting src/evaluate_v2.py


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
!python -m src.train_v2

Using device: cuda
Class weights: tensor([8.4837, 0.6159, 0.7946], device='cuda:0')

Starting training for 50 epochs...

Epoch:  1 | Val Accuracy: 0.5844 | LR: 0.000300
  --> Best model saved (val acc: 0.5844)
Epoch:  2 | Val Accuracy: 0.4257 | LR: 0.000300
Epoch:  3 | Val Accuracy: 0.5039 | LR: 0.000300
Epoch:  4 | Val Accuracy: 0.4983 | LR: 0.000300
Epoch:  5 | Val Accuracy: 0.5810 | LR: 0.000300
Epoch:  6 | Val Accuracy: 0.5844 | LR: 0.000150
Epoch:  7 | Val Accuracy: 0.5709 | LR: 0.000150
Epoch:  8 | Val Accuracy: 0.6302 | LR: 0.000150
  --> Best model saved (val acc: 0.6302)
Epoch:  9 | Val Accuracy: 0.5307 | LR: 0.000150
Epoch: 10 | Val Accuracy: 0.5922 | LR: 0.000150
Epoch: 11 | Val Accuracy: 0.6268 | LR: 0.000150
Epoch: 12 | Val Accuracy: 0.6391 | LR: 0.000150
  --> Best model saved (val acc: 0.6391)
Epoch: 13 | Val Accuracy: 0.5966 | LR: 0.000150
Epoch: 14 | Val Accuracy: 0.5966 | LR: 0.000150
Epoch: 15 | Val Accuracy: 0.6559 | LR: 0.000150
  --> Best model saved (val acc: 0.6

In [ ]:
!python -m src.evaluate_v2

Test Accuracy: 0.7262569832402235
              precision    recall  f1-score   support

         Low       0.37      0.54      0.44        35
      Medium       0.79      0.74      0.76       485
        High       0.70      0.73      0.71       375

    accuracy                           0.73       895
   macro avg       0.62      0.67      0.64       895
weighted avg       0.74      0.73      0.73       895



In [8]:
from google.colab import files
files.download("models/final_model_v2.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>